In [ ]:
import torch
import torchvision
import torch.nn as nn
import numpy as np

print("--- Deep Learning Environment Audit ---")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")

# Check if the cloud GPU is actively mapped to your notebook session
cuda_available = torch.cuda.is_available()
print(f"GPU Acceleration Available: {cuda_available}")

if cuda_available:
    print(f"Active GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Go to Runtime -> Change runtime type and select T4 GPU.")

--- Deep Learning Environment Audit ---
PyTorch Version: 2.11.0+cu128
Torchvision Version: 0.26.0+cu128
GPU Acceleration Available: True
Active GPU Device Name: Tesla T4


In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import GTSRB
from torch.utils.data import DataLoader, random_split

# 1. Define the structural pipeline for image transformation
transform_pipeline = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalize RGB channels
])

print("Downloading and parsing GTSRB dataset... (This may take a minute)")

# 2. Download the official training dataset split
full_dataset = GTSRB(root='./data', split='train', download=True, transform=transform_pipeline)

# 3. Split the training data into Train (80%) and Validation (20%) sets
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# 4. Construct DataLoaders to stream data in efficient batches
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print("\n--- Vision Data Pipeline Active ---")
print(f"Total Training Images: {len(train_dataset)}")
print(f"Total Validation Images: {len(val_dataset)}")

100%|██████████| 187M/187M [00:16<00:00, 11.2MB/s]




--- Vision Data Pipeline Active ---
Total Training Images: 21312
Total Validation Images: 5328

--- Vision Data Pipeline Active ---
Total Training Images: 21312
Total Validation Images: 5328


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TrafficSignCNN(nn.Module):
    def __init__(self):
        super(TrafficSignCNN, self).__init__()

        # Block 1: Input (3, 32, 32) -> Output (16, 16, 16)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 2: Input (16, 16, 16) -> Output (32, 8, 8)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)

        # Fully Connected Classifier Head
        self.fc1 = nn.Linear(32 * 8 * 8, 128)  # 32 channels * 8x8 image size = 2048 inputs
        self.fc2 = nn.Linear(128, 43)          # 43 target traffic sign classes

        self.relu = nn.ReLU()

    def forward(self, x):
        # Pass data through the convolutional features
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))

        # Flatten the spatial maps into a 1D vector for classification
        x = x.view(-1, 32 * 8 * 8)

        # Pass through the final classification layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model and move its parameters to the cloud GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_traffic = TrafficSignCNN().to(device)

print("--- CNN Architecture Compiled ---")
print(model_traffic)

--- CNN Architecture Compiled ---
TrafficSignCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2048, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=43, bias=True)
  (relu): ReLU()
)
--- CNN Architecture Compiled ---
TrafficSignCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2048, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=43, bias=True)
  (relu): ReLU()
)


In [ ]:
import torch.optim as optim

# 1. Define our grading rubric and optimization mechanic
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_traffic.parameters(), lr=0.001)

# 2. Set the number of complete passes through the dataset
epochs = 5

print("--- Launching Deep Learning Engine ---")
for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model_traffic.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        # Move tensor matrices directly onto the cloud GPU
        images, labels = images.to(device), labels.to(device)

        # Reset the weight gradients
        optimizer.zero_grad()

        # Forward pass + Loss tracking
        outputs = model_traffic(images)
        loss = criterion(outputs, labels)

        # Backward pass (Calculus chain rule)
        loss.backward()

        # Adjust internal weights
        optimizer.step()

        # Performance tracking metrics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct_train / total_train) * 100

    # --- VALIDATION PHASE ---
    model_traffic.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad(): # Turn off gradient calculations to save memory
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_traffic(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = (val_correct / val_total) * 100

    print(f"Epoch [{epoch+1}/{epochs}] -> Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("\n--- Training Pipeline Complete ---")

--- Launching Deep Learning Engine ---
Epoch [1/5] -> Train Loss: 1.9603 | Train Acc: 44.93% | Val Acc: 74.34%
Epoch [2/5] -> Train Loss: 0.5556 | Train Acc: 83.47% | Val Acc: 90.80%
Epoch [3/5] -> Train Loss: 0.2479 | Train Acc: 92.93% | Val Acc: 95.18%
Epoch [4/5] -> Train Loss: 0.1411 | Train Acc: 96.12% | Val Acc: 95.51%
Epoch [5/5] -> Train Loss: 0.1104 | Train Acc: 96.97% | Val Acc: 97.07%

--- Training Pipeline Complete ---
